# 07 · Check TRL + vLLM in one venv (before a paid run uses it)

Step 3 sampled GRPO answers with transformers' `generate`, which was most of each step's time. TRL
can sample with vLLM instead, but only with vLLM in the training venv. This notebook checks that
on free Kaggle GPUs, with a few GRPO steps on the same 4 real step 3 windows
(`docs/trl_vllm_check.md`):

1. **baseline:** transformers' generate, 4-bit Model A from the step 2 adapter, as step 3 would run here
2. **colocate:** vLLM inside the training process, on GPU 0, with Qwen3-0.6B and a fresh adapter
   (the H100 mode; a T4 cannot hold Model A twice, so this tests the code path, not memory)
3. **server:** `trl vllm-serve` with Model A on GPU 1, training Model A from the step 2 adapter in
   fp16 on GPU 0 (the Kaggle mode)

A check passes if it finishes, its answers earn some reward, and vLLM's token log probabilities
stay within 0.1 of the trainer's, which shows the trained weights reached vLLM.

**Kaggle:** Accelerator **GPU T4 x2** (not P100), Internet **on**, Add-ons → Secrets: attach
`HF_TOKEN`. "Save & Run All" runs it in the background.

**Time (estimate, not measured):** about 30-45 minutes: 10 to build the venv, then the three
checks. No Nebius call and no cost. The report goes to the Hub under `checks/trl_vllm/`.

In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "trl-vllm-check"
STEPS = 4                # GRPO steps per check


In [ ]:
import os, subprocess, sys
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
import site
with open(os.path.join(site.getsitepackages()[0], "credit_extractor.pth"), "w") as f:
    f.write(REPO + "\n")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["-r", "requirements.txt", "uv"], check=True)
uv = [sys.executable, "-m", "uv"]

# One venv with vLLM AND the training stack, which TRL's vLLM mode needs. vLLM's PyPI wheel
# needs CUDA 13; older drivers get its CUDA 12.9 build, as scripts/nebius_rlvr.sh does.
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
cuda_major = int(smi.split("CUDA Version:")[1].split(".")[0]) if "CUDA Version:" in smi else 13
vllm_pkgs = (["--torch-backend=auto", "-r", "requirements-gpu.txt"] if cuda_major >= 13 else
             ["--torch-backend=cu129", "--extra-index-url", "https://wheels.vllm.ai/0.32.0/cu129",
              "vllm==0.32.0+cu129"])
VENV = "/tmp/rl-vllm-env"; RPY = VENV + "/bin/python"
if not os.path.exists(RPY):
    subprocess.run(uv + ["venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(uv + ["pip", "install", "-q", "--python", RPY, *vllm_pkgs,
                             "-r", "requirements-rl-vllm.txt", "-r", "requirements.txt", "wrapt"],
                       capture_output=True, text=True)
    # a resolver conflict is itself the first finding: print it whole
    assert r.returncode == 0, "combined install failed:\n" + r.stderr[-4000:]
CHECK = ("import torch, vllm, trl, transformers, peft\n"
         "print('combined venv: vllm', vllm.__version__, '| trl', trl.__version__, '| transformers',"
         " transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__,"
         " 'cuda', torch.version.cuda, 'gpus', torch.cuda.device_count())")
r = subprocess.run([RPY, "-c", CHECK], capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])
assert r.returncode == 0, "combined venv not usable"


In [ ]:
from src.platform import setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (inputs and the report use the Hub)"
assert len(env.gpus) >= 2, "pick GPU T4 x2: the server check needs a second GPU"


In [ ]:
# baseline, then colocate, then server; each check's log is under work/checks/trl_vllm/logs/
from src import check_vllm
report = check_vllm.run_all(env, RPY, steps=STEPS)


In [ ]:
print(f"session wall time: {env.session_minutes():.1f} min")
